In [51]:
#################################################
# 1. Load Credentials from .env
#################################################

import os
from dotenv import load_dotenv
import requests
from requests.auth import HTTPBasicAuth
import pandas as pd

load_dotenv()

USER = os.getenv("DBREPO_USER")
PASSWORD = os.getenv("DBREPO_PASSWORD")

print("Credentials loaded.")

Credentials loaded.


In [52]:
#################################################
# 2. DBRepo Client Setup
#################################################

from dbrepo.RestClient import RestClient

BASE_URL = "https://test.dbrepo.tuwien.ac.at"

client = RestClient(
    BASE_URL,
    username=USER,
    password=PASSWORD
)

print("DBRepo client initialized.")

DBRepo client initialized.


In [53]:
#################################################
# 3. Create / Get Database
#################################################

DB_NAME = "data_stew_grp22_air_quality"
CONTAINER_ID = "6cfb3b8e-1792-4e46-871a-f3d103527203"

databases = client.get_databases()
DB_ID = None

# Check if DB already exists
for db in databases:

    if db.name == DB_NAME:

        DB_ID = db.id
        print("Using existing DB:", DB_ID)
        break

# Create database if it does not exist
if DB_ID is None:

    print("Creating database...")

    try:

        client.create_database(
            DB_NAME,
            CONTAINER_ID,
            is_public=False,
            is_schema_public=False
        )

    except Exception as e:

        print("Database creation returned an exception:")
        print(e)

    # Re-fetch databases
    databases = client.get_databases()

    for db in databases:

        if db.name == DB_NAME:

            DB_ID = db.id
            print("Database found after creation:", DB_ID)
            break

# Final validation
if DB_ID is None:

    raise Exception("Database could not be created or retrieved.")

print("Final DB ID:", DB_ID)

Using existing DB: 81d82941-cae0-4cba-b27d-1bd883dd713a
Final DB ID: 81d82941-cae0-4cba-b27d-1bd883dd713a


In [54]:
#################################################
# 4. Clean Existing Tables
#################################################

tables = client.get_tables(DB_ID)

if len(tables) == 0:
    print("Database already clean.")
else:
    for t in tables:
        print("Deleting:", t.name)
        client.delete_table(DB_ID, t.id)

Deleting: t_measurement
Deleting: t_time
Deleting: t_station


In [55]:
#################################################
# 5. Create Table: t_station
#################################################

tables = client.get_tables(DB_ID)
table_names = [t.name for t in tables]

if "t_station" in table_names:

    print("t_station already exists. Skipping.")

else:

    df_station = pd.DataFrame({
        "station_id": pd.Series([1], dtype="int64"),
        "station_name": pd.Series(["dummy_station"], dtype="object"),
        "latitude": pd.Series([0.0], dtype="float"),
        "longitude": pd.Series([0.0], dtype="float")
    })

    df_station = df_station.set_index("station_id")

    table_station = client.create_table(
        DB_ID,
        "t_station",
        description="Metadata about air quality monitoring stations including station names and geographic coordinates.",
        is_public=False,
        is_schema_public=False,
        dataframe=df_station,
        with_data=False
    )

    print("Created:", table_station.name)

Created: t_station


In [56]:
#################################################
# 6. Create Table: t_time
#################################################

tables = client.get_tables(DB_ID)
table_names = [t.name for t in tables]

if "t_time" in table_names:

    print("t_time already exists. Skipping.")

else:

    df_time = pd.DataFrame({
        "time_id": pd.Series([1], dtype="int64"),
        "date": ["2013-01-01"],
        "hour": pd.Series([12], dtype="int64")
    })

    df_time = df_time.set_index("time_id")

    table_time = client.create_table(
        DB_ID,
        "t_time",
        description="Temporal dimension table containing date and hour information for air quality measurements.",
        is_public=False,
        is_schema_public=False,
        dataframe=df_time,
        with_data=False
    )

    print("Created:", table_time.name)

Created: t_time


In [57]:
#################################################
# 7. Create Table: t_measurement
#################################################

tables = client.get_tables(DB_ID)
table_names = [t.name for t in tables]

if "t_measurement" in table_names:

    print("t_measurement already exists. Skipping.")

else:

    df_measurement = pd.DataFrame({

        "measurement_id": pd.Series([1], dtype="int64"),
        "station_id": pd.Series([1], dtype="int64"),
        "time_id": pd.Series([1], dtype="int64"),

        "SO2": pd.Series([0.0], dtype="float"),
        "NO": pd.Series([0.0], dtype="float"),
        "NO2": pd.Series([0.0], dtype="float"),
        "CO": pd.Series([0.0], dtype="float"),
        "PM10": pd.Series([0.0], dtype="float"),
        "O3": pd.Series([0.0], dtype="float"),
        "PM25": pd.Series([0.0], dtype="float"),

        "wind_direction": pd.Series([0.0], dtype="float"),
        "wind_speed": pd.Series([0.0], dtype="float"),

        "temperature": pd.Series([0.0], dtype="float"),
        "humidity": pd.Series([0.0], dtype="float"),
        "pressure": pd.Series([0.0], dtype="float"),
        "solar_radiation": pd.Series([0.0], dtype="float"),
        "rain": pd.Series([0.0], dtype="float"),

        "BEN": pd.Series([0.0], dtype="float"),
        "TOL": pd.Series([0.0], dtype="float"),
        "MXIL": pd.Series([0.0], dtype="float")
    })

    df_measurement = df_measurement.set_index("measurement_id")

    table_measurement = client.create_table(
        DB_ID,
        "t_measurement",
        description="Air quality and meteorological measurements linked to monitoring stations and timestamps.",
        is_public=False,
        is_schema_public=False,
        dataframe=df_measurement,
        with_data=False
    )

    print("Created:", table_measurement.name)

Created: t_measurement


In [58]:
#################################################
# 8 Show DB, Tables, and Columns
#################################################

# Get database metadata
db = client.get_database(DB_ID)

print(f"\nDatabase: {db.name}")
print(f"Internal Name: {db.internal_name}")

if hasattr(db, "owner"):
    print(f"Owner: {db.owner.username}")
elif hasattr(db, "owned_by"):
    print(f"Owner: {db.owned_by}")

print(f"Public: {db.is_public}")
print(f"Schema Public: {db.is_schema_public}")

# Get tables
tables = client.get_tables(DB_ID)

print(f"\nTables in DB '{db.name}':")

for t in tables:

    print("\n" + "-" * 50)
    print(f"Table: {t.name}")

    table_details = client.get_table(DB_ID, t.id)

    if hasattr(table_details, "description"):
        print(f"Description: {table_details.description}")

    print("\nColumns:")

    for col in table_details.columns:

        column_info = f"  - {col.name} ({col.type})"

        if hasattr(col, "is_null_allowed"):
            column_info += f" | Nullable: {col.is_null_allowed}"

        print(column_info)


Database: data_stew_grp22_air_quality
Internal Name: data_stew_grp22_air_quality_gvui
Owner: 12450741
Public: True
Schema Public: True

Tables in DB 'data_stew_grp22_air_quality':

--------------------------------------------------
Table: t_measurement
Description: Air quality and meteorological measurements linked to monitoring stations and timestamps.

Columns:
  - measurement_id (ColumnType.BIGINT) | Nullable: False
  - station_id (ColumnType.BIGINT) | Nullable: False
  - time_id (ColumnType.BIGINT) | Nullable: False
  - SO2 (ColumnType.DECIMAL) | Nullable: False
  - NO (ColumnType.DECIMAL) | Nullable: False
  - NO2 (ColumnType.DECIMAL) | Nullable: False
  - CO (ColumnType.DECIMAL) | Nullable: False
  - PM10 (ColumnType.DECIMAL) | Nullable: False
  - O3 (ColumnType.DECIMAL) | Nullable: False
  - PM25 (ColumnType.DECIMAL) | Nullable: False
  - wind_direction (ColumnType.DECIMAL) | Nullable: False
  - wind_speed (ColumnType.DECIMAL) | Nullable: False
  - temperature (ColumnType.DECIM

In [59]:
#################################################
# 9. Create Database Identifier
#################################################

from dbrepo.api.dto import (
    IdentifierType,
    License,
    CreateIdentifierCreator,
    CreateIdentifierTitle,
    CreateIdentifierDescription,
    DescriptionType,
)

identifier = client.create_identifier(
    DB_ID,
    type=IdentifierType.DATABASE,

    creators=[
        CreateIdentifierCreator(
            creator_name="Ayuntamiento de Gijón"
        )
    ],

    titles=[
        CreateIdentifierTitle(
            title="Air Quality Dataset – Gijón (2013)"
        )
    ],

    descriptions=[
        CreateIdentifierDescription(
            description="Air quality measurements from Gijón for 2013.",
            type=DescriptionType.ABSTRACT,
        )
    ],

    licenses=[
        License(
            identifier="CC-BY-4.0",
            uri="https://creativecommons.org/licenses/by/4.0/",
            description="Creative Commons Attribution 4.0 International",
        )
    ],

    funders=[],
    related_identifiers=[],

    publisher="Ayuntamiento de Gijón",
    publication_year=2013,
)

print(identifier)

id='319777bd-0f81-42ab-be5d-77b1a74fbae6' database_id='81d82941-cae0-4cba-b27d-1bd883dd713a' links=Links(self='/api/v1/identifier/319777bd-0f81-42ab-be5d-77b1a74fbae6', self_html='/pid/319777bd-0f81-42ab-be5d-77b1a74fbae6', data=None) type=<IdentifierType.DATABASE: 'database'> owner=UserBrief(username='12450741', id=None, name=None, orcid=None, qualified_name=None, given_name=None, family_name=None) status=<IdentifierStatusType.DRAFT: 'draft'> publication_year=2013 publisher='Ayuntamiento de Gijón' creators=[Creator(id='b8ec59e2-7f1f-43b3-af23-2ff0d5218ed5', creator_name='Ayuntamiento de Gijón', firstname=None, lastname=None, name_type=None, name_identifier=None, name_identifier_scheme=None, name_identifier_scheme_uri=None, affiliation=None, affiliation_identifier=None, affiliation_identifier_scheme=None, affiliation_identifier_scheme_uri=None)] titles=[IdentifierTitle(id='ca5cf43a-0a9e-41e9-a3c5-d40116ad856f', title='Air Quality Dataset – Gijón (2013)', language=None, type=None)] desc

In [61]:
#################################################
# 10. Inspect Database Metadata
#################################################

db = client.get_database(DB_ID)

print("Database Metadata")
print("-" * 50)

print("ID:", db.id)
print("Name:", db.name)
print("Internal Name:", db.internal_name)
print("Description:", db.description)

# Safe owner handling
if hasattr(db, "owner"):
    print("Owner:", db.owner.username)
elif hasattr(db, "owned_by"):
    print("Owner:", db.owned_by)

print("Public:", db.is_public)
print("Schema Public:", db.is_schema_public)

# Identifiers
if hasattr(db, "identifiers"):
    print("\nIdentifiers:")
    print(db.identifiers)

# Contact
if hasattr(db, "contact"):
    print("\nContact:")
    print(db.contact)

Database Metadata
--------------------------------------------------
ID: 81d82941-cae0-4cba-b27d-1bd883dd713a
Name: data_stew_grp22_air_quality
Internal Name: data_stew_grp22_air_quality_gvui
Description: None
Owner: 12450741
Public: True
Schema Public: True

Identifiers:
[Identifier(id='319777bd-0f81-42ab-be5d-77b1a74fbae6', database_id='81d82941-cae0-4cba-b27d-1bd883dd713a', links=Links(self='/api/v1/identifier/319777bd-0f81-42ab-be5d-77b1a74fbae6', self_html='/pid/319777bd-0f81-42ab-be5d-77b1a74fbae6', data=None), type=<IdentifierType.DATABASE: 'database'>, owner=UserBrief(username='12450741', id=None, name=None, orcid=None, qualified_name=None, given_name=None, family_name=None), status=<IdentifierStatusType.DRAFT: 'draft'>, publication_year=2013, publisher='Ayuntamiento de Gijón', creators=[Creator(id='b8ec59e2-7f1f-43b3-af23-2ff0d5218ed5', creator_name='Ayuntamiento de Gijón', firstname=None, lastname=None, name_type=None, name_identifier=None, name_identifier_scheme=None, name_i

In [63]:
#################################################
# 11. Inspect Database Metadata and PID
#################################################

db = client.get_database(DB_ID)

print("Database Metadata")
print("-" * 50)

print("ID:", db.id)

print("Name:", db.name)

print("Internal Name:", db.internal_name)

print("Description:", db.description)

# Safe owner handling
if hasattr(db, "owner") and db.owner is not None:

    print("Owner:", db.owner.username)

elif hasattr(db, "owned_by"):

    print("Owner:", db.owned_by)

print("Public:", db.is_public)

print("Schema Public:", db.is_schema_public)

print("\nPersistent Identifier (PID)")
print("-" * 50)

# Use already-created identifier object
print("Identifier ID:", identifier.id)

print("DOI:", identifier.doi)

print("Publisher:", identifier.publisher)

print("Publication Year:", identifier.publication_year)

print("Status:", identifier.status)

print("License:")

for lic in identifier.licenses:

    print("-", lic.identifier)

    print(" ", lic.uri)

Database Metadata
--------------------------------------------------
ID: 81d82941-cae0-4cba-b27d-1bd883dd713a
Name: data_stew_grp22_air_quality
Internal Name: data_stew_grp22_air_quality_gvui
Description: None
Owner: 12450741
Public: True
Schema Public: True

Persistent Identifier (PID)
--------------------------------------------------
Identifier ID: 319777bd-0f81-42ab-be5d-77b1a74fbae6
DOI: 10.82556/0h29-hr36
Publisher: Ayuntamiento de Gijón
Publication Year: 2013
Status: IdentifierStatusType.DRAFT
License:
- CC-BY-4.0
  https://creativecommons.org/licenses/by/4.0/legalcode


In [64]:
#################################################
# 12. Inspect Table Metadata
#################################################

tables = client.get_tables(DB_ID)

for t in tables:

    print("\n" + "=" * 60)
    print(f"Table: {t.name}")
    print("=" * 60)

    table_meta = client.get_table(DB_ID, t.id)

    print("ID:", table_meta.id)
    print("Name:", table_meta.name)
    print("Internal Name:", table_meta.internal_name)
    print("Description:", table_meta.description)

    # Safe owner handling
    if hasattr(table_meta, "owner"):
        print("Owner:", table_meta.owner.username)
    elif hasattr(table_meta, "owned_by"):
        print("Owner:", table_meta.owned_by)

    print("Public:", table_meta.is_public)
    print("Schema Public:", table_meta.is_schema_public)

    # Row count if available
    if hasattr(table_meta, "num_rows"):
        print("Rows:", table_meta.num_rows)

    # Identifiers
    if hasattr(table_meta, "identifiers"):
        print("Identifiers:", table_meta.identifiers)

    print("\nColumns:")

    for col in table_meta.columns:

        print(f"  - {col.name}")

        if hasattr(col, "type"):
            print(f"      Type: {col.type}")

        if hasattr(col, "is_null_allowed"):
            print(f"      Nullable: {col.is_null_allowed}")

        if hasattr(col, "description"):
            print(f"      Description: {col.description}")

        if hasattr(col, "concept"):
            print(f"      Concept URI: {col.concept}")

        if hasattr(col, "unit"):
            print(f"      Unit URI: {col.unit}")


Table: t_measurement
ID: cac30024-f374-4143-a86a-8c8196001375
Name: t_measurement
Internal Name: t_measurement
Description: Air quality and meteorological measurements linked to monitoring stations and timestamps.
Owner: 12450741
Public: False
Schema Public: False
Rows: 42804
Identifiers: []

Columns:
  - measurement_id
      Type: ColumnType.BIGINT
      Nullable: False
      Description: None
      Concept URI: None
      Unit URI: None
  - station_id
      Type: ColumnType.BIGINT
      Nullable: False
      Description: None
      Concept URI: None
      Unit URI: None
  - time_id
      Type: ColumnType.BIGINT
      Nullable: False
      Description: None
      Concept URI: None
      Unit URI: None
  - SO2
      Type: ColumnType.DECIMAL
      Nullable: False
      Description: None
      Concept URI: None
      Unit URI: None
  - NO
      Type: ColumnType.DECIMAL
      Nullable: False
      Description: None
      Concept URI: None
      Unit URI: None
  - NO2
      Type: ColumnTy